<div class="blog-language-switch" role="group" aria-label="文章语言">
<a href="../../Deep-Learning/21-evaluation-interpretability-robustness-responsibility.html" lang="en" hreflang="en">English</a>
<span aria-current="page">中文</span>
</div>

[返回深度学习总览](Deep-Learning.html)

## **评估、可解释性、鲁棒性与负责任的深度学习** {#evaluation-interpretability-robustness-responsibility}

评估是一套由证据支持的论证：某个确定的模型产物，在明确的数据和运行条件下，适合某项已经声明的决策。平均 benchmark 分数只是其中一个前提。可靠应用还需要经过校准的不确定性、已知失败边界、压力测试、subgroup 证据、数据血缘、隐私分析，以及假设失效时的响应方式。可解释性只有在检验具体假设时才能支撑这套论证；一张好看的 heatmap 本身不能建立信任。

本章使用 scikit-learn 收录的 [UCI Optical Recognition of Handwritten Digits](https://doi.org/10.24432/C50P49) 数据集，采用 **CC BY 4.0** 许可。数字 0–8 构成九分类的分布内（ID）任务；所有数字 9 样本完全排除在模型拟合之外，作为语义分布外（OOD）集合。固定的数据划分和同一个小型 CNN 贯穿后续校准、probe、attribution、对抗、隐私与 subgroup 实验。该受控设计便于比较机制，但数字 9 只是一种较简单的 OOD，数据集也不包含人口属性。

![数字零到八构成 ID 任务，数字九被保留为 OOD。](assets/dl21-id-ood-grid.png){fig-align="center" width="68%" fig-alt="图中展示十张手写数字小图，标签零到八属于分布内，标签九被标记为分布外。"}

*数据来源：Alpaydin 与 Kaynak，[UCI Optical Recognition of Handwritten Digits](https://doi.org/10.24432/C50P49)，CC BY 4.0。样本来自 scikit-learn 文档记录的 `load_digits` 副本。*

<details>
<summary><strong>PyTorch：建立整章共享的 ID/OOD 评估工作负载</strong></summary>

```python
import hashlib
import io
import json
import random

import numpy as np
import sklearn
import torch
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.set_num_threads(1)


def seed_everything(seed=2121):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


seed_everything()
digits = load_digits()
all_ids = np.arange(len(digits.data))
id_ids = all_ids[digits.target != 9]
ood_ids = all_ids[digits.target == 9]
train_ids, remaining_ids = train_test_split(
    id_ids, test_size=0.30, stratify=digits.target[id_ids], random_state=2121
)
val_ids, test_ids = train_test_split(
    remaining_ids,
    test_size=0.50,
    stratify=digits.target[remaining_ids],
    random_state=2121,
)

# UCI fixes intensities to 0..16; division by 16 does not fit a test statistic.
images = torch.tensor(digits.images / 16.0, dtype=torch.float32)[:, None, :, :]
targets = torch.tensor(digits.target, dtype=torch.long)
train_dataset = TensorDataset(images[train_ids], targets[train_ids])
val_dataset = TensorDataset(images[val_ids], targets[val_ids])
test_dataset = TensorDataset(images[test_ids], targets[test_ids])
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    generator=torch.Generator().manual_seed(2121),
)
val_loader = DataLoader(val_dataset, batch_size=128)
test_loader = DataLoader(test_dataset, batch_size=128)


class TinyDigitCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(32 * 4 * 4, 64)
        self.dropout = nn.Dropout(0.15)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 9)

    def forward_intermediates(self, x):
        conv1 = F.gelu(self.conv1(x))
        conv2 = F.gelu(self.conv2(conv1))
        pooled = F.avg_pool2d(conv2, kernel_size=2).flatten(1)
        embedding = F.gelu(self.fc1(pooled))
        hidden = F.gelu(self.fc2(self.dropout(embedding)))
        logits = self.fc3(hidden)
        return logits, {"conv1": conv1, "conv2": conv2, "embedding": embedding, "hidden": hidden}

    def logits_from_embedding(self, embedding):
        hidden = F.gelu(self.fc2(self.dropout(embedding)))
        return self.fc3(hidden)

    def forward(self, x):
        return self.forward_intermediates(x)[0]


def batched_logits(model, inputs, batch_size=128):
    model.eval()
    outputs = []
    with torch.inference_mode():
        for start in range(0, len(inputs), batch_size):
            outputs.append(model(inputs[start : start + batch_size]))
    return torch.cat(outputs)


seed_everything()
model = TinyDigitCNN()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
for _ in range(18):
    model.train()
    for x, y in train_loader:
        loss = F.cross_entropy(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

id_test_x, id_test_y = images[test_ids], targets[test_ids]
ood_x = images[ood_ids]
id_test_logits = batched_logits(model, id_test_x)
baseline_accuracy = float(id_test_logits.argmax(1).eq(id_test_y).float().mean())
assert not set(train_ids) & set(test_ids)
assert set(targets[train_ids].tolist()) == set(range(9)) and set(targets[ood_ids].tolist()) == {9}
assert baseline_accuracy > 0.93
print({"split": (len(train_ids), len(val_ids), len(test_ids)), "OOD": len(ood_ids), "ID_accuracy": round(baseline_accuracy, 3)})
```

</details>

模型被有意控制在较小规模，OOD 边界也被显式定义。结果只验证该工作负载上的评估流程；它不能证明模型已经适合真实世界手写识别、人口公平性、隐私保证或对抗安全。


### **跨模态与任务的评估** {#evaluation-across-modalities-tasks}

指标只有在任务、分析单位、数据分布和决策成本都明确时才有意义。分类可能需要 accuracy、macro-F1、逐类别 recall、校准和 abstention 行为；检索需要 recall@(k)、MRR 或 nDCG 等排序指标，并定义 relevance set；分割在 object 或 pixel 层面报告 IoU/Dice；生成任务需要多个 reference 或人工判断，因为词汇重合、事实性、多样性与偏好不是同一属性；representation learning 依赖下游 probe 或 transfer protocol；强化学习则需要 return 分布、约束违反情况和跨 seed 评估。

![从单元测试延伸到系统与决策背景的评估栈。](assets/dl21-evaluation-stack.svg){fig-align="center" width="76%" fig-alt="五层评估依次覆盖单元测试、数据集、slice、压力测试和系统证据，并统一落在决策背景中。"}

评估单位必须阻止泄漏：医学影像应按 patient 而不是 scan 划分，语音按 speaker 而不是 utterance，交互数据按 user 而不是 event；如果部署目标是未来，还应按时间划分。在接触测试集之前冻结预处理和阈值。即使测试行从未参与 gradient descent，反复针对公开 benchmark 调参也会让它变成训练反馈。

![分布内 Digits 测试集的 confusion matrix。](assets/dl21-confusion-matrix.png){fig-align="center" width="62%" fig-alt="九乘九 confusion matrix 展示数字零到八的真实类别与预测类别计数。"}

<details>
<summary><strong>Python：组合总体、类别与 selective-prediction 证据</strong></summary>

```python
id_predictions = id_test_logits.argmax(dim=1)
id_probabilities = id_test_logits.softmax(dim=1)
id_confidence = id_probabilities.max(dim=1).values
macro_f1 = f1_score(id_test_y.numpy(), id_predictions.numpy(), average="macro")
matrix = confusion_matrix(id_test_y.numpy(), id_predictions.numpy(), labels=list(range(9)))
per_class_recall = matrix.diagonal() / matrix.sum(axis=1).clip(min=1)

# Selective prediction: abstain on the lowest-confidence 20 percent.
keep = id_confidence >= torch.quantile(id_confidence, 0.20)
selective_accuracy = float(id_predictions[keep].eq(id_test_y[keep]).float().mean())
coverage = float(keep.float().mean())
assert matrix.sum() == len(test_ids) and 0 < coverage <= 1
print({
    "accuracy": round(baseline_accuracy, 3),
    "macro_F1": round(macro_f1, 3),
    "worst_class_recall": round(float(per_class_recall.min()), 3),
    "selective_accuracy": round(selective_accuracy, 3),
    "coverage": round(coverage, 3),
})
```

</details>

Selective accuracy 并不是免费提升：abstained 样本被转移到另一个流程，而该流程的成本与质量同样需要测量。当采样或优化方差不可忽略时，应报告 uncertainty interval 或多个 seed，并保留原始预测，让后续 slice 分析不依赖四舍五入后的汇总数字。


### **校准与预测不确定性** {#calibration-predictive-uncertainty}

Accuracy 询问 top class 是否正确；calibration 询问信心能否对应经验频率。对一组 confidence 为 0.8 的已校准预测，正确率应当接近 80%。Negative log-likelihood 和 Brier score 是评价完整概率分布的 proper scoring rule。Expected calibration error（ECE）对预测分箱后计算

$$\mathrm{ECE}=\sum_{m=1}^{M}\frac{|B_m|}{n}\left|\operatorname{acc}(B_m)-\operatorname{conf}(B_m)\right|.$$

ECE 容易解释，但依赖分箱方式，也可能掩盖 class-conditional error。Reliability diagram 保留了更多结构。Temperature scaling 在 validation logits 上学习一个正标量 (T)，并使用 (\operatorname{softmax}(z/T)) 预测；它改变 confidence，但不改变 argmax。[Guo 等人的校准研究](https://proceedings.mlr.press/v70/guo17a.html)把它确立为有力的 post-hoc baseline。

![Temperature scaling 前后的 reliability diagram。](assets/dl21-calibration.png){fig-align="center" width="64%" fig-alt="可靠性图将原始与 temperature-scaled confidence 对照经验 accuracy，并给出完美校准对角线。"}

预测不确定性有不同来源。Aleatoric uncertainty 来自观测本身无法消除的歧义；epistemic uncertainty 来自模型参数或未被支持的区域。Softmax entropy 会混合多种效应，不能自动等同于 epistemic uncertainty。Ensemble 和近似 Bayesian 方法可以暴露模型分歧，但结论仍受其训练和近似假设约束。

<details>
<summary><strong>PyTorch：拟合 temperature scaling 并与 MC-dropout disagreement 比较</strong></summary>

```python
def expected_calibration_error(logits, labels, bins=10):
    probabilities = logits.softmax(dim=1)
    confidence, prediction = probabilities.max(dim=1)
    correctness = prediction.eq(labels).float()
    edges = torch.linspace(0, 1, bins + 1)
    ece = torch.tensor(0.0)
    for lower, upper in zip(edges[:-1], edges[1:]):
        mask = (confidence > lower) & (confidence <= upper)
        if mask.any():
            ece += mask.float().mean() * (correctness[mask].mean() - confidence[mask].mean()).abs()
    return float(ece)


validation_logits = batched_logits(model, images[val_ids])
validation_labels = targets[val_ids]
temperature_grid = torch.linspace(0.5, 4.0, 141)
validation_nll = torch.tensor([
    F.cross_entropy(validation_logits / value, validation_labels) for value in temperature_grid
])
temperature = float(temperature_grid[validation_nll.argmin()])
raw_ece = expected_calibration_error(id_test_logits, id_test_y)
calibrated_ece = expected_calibration_error(id_test_logits / temperature, id_test_y)
raw_test_nll = float(F.cross_entropy(id_test_logits, id_test_y))
calibrated_test_nll = float(F.cross_entropy(id_test_logits / temperature, id_test_y))

# Enable only Dropout at test time; this model has no BatchNorm state to disturb.
model.train()
with torch.inference_mode():
    mc_probabilities = torch.stack([model(id_test_x[:64]).softmax(1) for _ in range(24)])
model.eval()
predictive_entropy = -(mc_probabilities.mean(0) * mc_probabilities.mean(0).clamp_min(1e-8).log()).sum(1)
disagreement = mc_probabilities.var(0).sum(1)
assert temperature > 0 and predictive_entropy.shape == disagreement.shape == (64,)
print({
    "temperature": round(temperature, 3),
    "ECE_raw/calibrated": (round(raw_ece, 3), round(calibrated_ece, 3)),
    "NLL_raw/calibrated": (round(raw_test_nll, 3), round(calibrated_test_nll, 3)),
    "mean_MC_disagreement": round(float(disagreement.mean()), 5),
})
```

</details>

领域、class prior、数值精度或模型发生变化后，都必须重新检查 calibration。全局校准良好的模型仍可能在关键 subgroup 上失准。不确定性方法应根据它要驱动的决策来选择：abstention、active learning、OOD routing 和 risk-sensitive ranking 所需的验证证据并不相同。


### **分布外检测** {#out-of-distribution-detection}

OOD 是相对于训练分布和预期任务定义的。新的背景样式属于 covariate shift；数字 9 这样的未知类别属于 semantic shift；标签规则变化属于 concept shift。检测、泛化与适应是不同目标：detector 可以标记输入却不知道如何分类；robust model 也可能在没有显式检测的 shift 下保持准确。

Maximum softmax probability（MSP）是有用 baseline：较低最大 confidence 暗示 OOD，但 discriminative network 可能在远离训练数据的位置仍然高度自信。对于 logits (z_k(x)) 与 temperature (T)，energy score 为

$$E(x)=-T\log\sum_{k=1}^{K}\exp(z_k(x)/T).$$

ID 输入通常具有更低 energy。[Energy-based OOD 研究](https://proceedings.neurips.cc/paper/2020/hash/f5496252609c43eb8a3d147ab9b9c006-Abstract.html)给出了不同于 MSP 的形式化方法。两种 score 都不是“输入属于 OOD 的概率”。

![ID 数字与保留数字九的 energy-score 分布。](assets/dl21-ood-energy.png){fig-align="center" width="66%" fig-alt="重叠直方图比较分布内数字零到八和保留数字九的 energy score。"}

<details>
<summary><strong>PyTorch：在保留数字上比较 MSP 与 energy detection</strong></summary>

```python
ood_logits = batched_logits(model, ood_x)
id_msp_score = 1.0 - id_test_logits.softmax(1).max(1).values
ood_msp_score = 1.0 - ood_logits.softmax(1).max(1).values
id_energy = -torch.logsumexp(id_test_logits, dim=1)
ood_energy = -torch.logsumexp(ood_logits, dim=1)


def ood_auc(id_scores, shifted_scores):
    labels = np.concatenate([np.zeros(len(id_scores)), np.ones(len(shifted_scores))])
    scores = torch.cat([id_scores, shifted_scores]).numpy()
    return roc_auc_score(labels, scores)


msp_ood_auc = ood_auc(id_msp_score, ood_msp_score)
energy_ood_auc = ood_auc(id_energy, ood_energy)
validation_energy = -torch.logsumexp(validation_logits, dim=1)
ood_energy_threshold = float(torch.quantile(validation_energy, 0.95))
id_false_positive_rate = float((id_energy > ood_energy_threshold).float().mean())
ood_true_positive_rate = float((ood_energy > ood_energy_threshold).float().mean())
assert 0 <= msp_ood_auc <= 1 and 0 <= energy_ood_auc <= 1
print({
    "MSP_AUROC": round(msp_ood_auc, 3),
    "energy_AUROC": round(energy_ood_auc, 3),
    "validation_threshold_ID_FPR": round(id_false_positive_rate, 3),
    "digit9_TPR": round(ood_true_positive_rate, 3),
})
```

</details>

AUROC 与阈值无关，但在不现实的类别比例下可能显得过于乐观。还应报告指定 TPR 下的 FPR、precision-recall curve 与真实负载 prevalence。需要测试多个 near-OOD 和 far-OOD family；针对数字 9 调整的 detector 可能无法识别模糊、空白输入或新采集设备。阈值必须在 validation data 上确定，并绑定 abstain、route 或收集复核等操作。


### **特征与表征 Probe** {#feature-representation-probing}

Probe 冻结 representation，并为关注属性训练受控 readout。它询问该 representation 中的信息在既定 probe capacity 与 data budget 下是否**可解码**。比较不同层有助于观察类别、语法、几何或其他属性何时变得线性可访问；但这不能证明原模型使用了该信息，也不能证明解码方向具有因果性。

![冻结网络向受控 probe 提供表征，并明确限制结论边界。](assets/dl21-probing.svg){fig-align="center" width="74%" fig-alt="多个冻结层的 representation 输入同一种受控 linear probe，最终警告 decodability 不等于 causality。"}

Probe 设计必须控制数据划分、类别平衡、维度、regularization 与 probe capacity。高容量 nonlinear probe 可能从微弱痕迹重新学会任务；高维随机 representation 也可能具有惊人的可分性。Selectivity baseline 会把真实属性与随机标签或匹配的 control task 比较。

<details>
<summary><strong>PyTorch 与 scikit-learn：比较不同 CNN 层的线性可解码性</strong></summary>

```python
def collect_representations(model, inputs, batch_size=128):
    model.eval()
    storage = {"pixels": [], "conv1": [], "conv2": [], "embedding": []}
    with torch.inference_mode():
        for start in range(0, len(inputs), batch_size):
            batch = inputs[start : start + batch_size]
            _, parts = model.forward_intermediates(batch)
            storage["pixels"].append(batch.flatten(1))
            storage["conv1"].append(F.avg_pool2d(parts["conv1"], 2).flatten(1))
            storage["conv2"].append(F.avg_pool2d(parts["conv2"], 2).flatten(1))
            storage["embedding"].append(parts["embedding"])
    return {name: torch.cat(values).numpy() for name, values in storage.items()}


train_representations = collect_representations(model, images[train_ids])
test_representations = collect_representations(model, id_test_x)
probe_scores = {}
for layer_name in train_representations:
    probe = LogisticRegression(max_iter=1200, C=1.0, random_state=2121)
    probe.fit(train_representations[layer_name], targets[train_ids].numpy())
    probe_scores[layer_name] = probe.score(test_representations[layer_name], id_test_y.numpy())

assert set(probe_scores) == {"pixels", "conv1", "conv2", "embedding"}
print({name: round(score, 3) for name, score in probe_scores.items()})
```

</details>

Linear score 较弱的层仍可能包含有用的非线性信息；较高分数也可能来自 nuisance correlation。只有与 intervention、transfer test 和 negative control 配合时，probe 结果才更有价值，不能把它当作 representation learning 的完整解释。


### **基于梯度的 Attribution** {#gradient-based-attribution}

Input-gradient attribution 计算选定 score 对无穷小 feature 变化的响应。对 target logit (f_c(x))，saliency 常写为 (S_i=|\partial f_c(x)/\partial x_i|)。它是局部、model-specific 且计算便宜的方法。饱和 nonlinear function 可能让重要 feature 的 gradient 很小，而轻微输入变化也可能产生视觉上不稳定的 map。

SmoothGrad 对带噪输入的 gradient 求平均：(\bar{S}(x)=\frac{1}{N}\sum_n S(x+\epsilon_n))。它可以减少视觉噪声，却不会改变底层模型。Gradient times input 加入当前 feature magnitude，但零 reference 未必具有语义。Target 必须明确：predicted logit、true-class logit、margin、probability 与 loss 回答的是不同问题。

![同一预测的输入、原始 gradient saliency 与 SmoothGrad。](assets/dl21-saliency.png){fig-align="center" width="66%" fig-alt="三个 panel 分别展示手写数字、绝对 input-gradient map 和经过平均的平滑 gradient map。"}

<details>
<summary><strong>PyTorch：计算 saliency 并用 feature deletion 检验</strong></summary>

```python
correct_positions = torch.where(id_predictions.eq(id_test_y))[0]
representative_position = int(correct_positions[id_confidence[correct_positions].argmax()])
representative_x = id_test_x[representative_position : representative_position + 1]
representative_target = int(id_predictions[representative_position])

gradient_input = representative_x.clone().requires_grad_(True)
target_score = model(gradient_input)[0, representative_target]
saliency = torch.autograd.grad(target_score, gradient_input)[0].abs()

noise_generator = torch.Generator().manual_seed(2121)
smooth_gradients = []
for _ in range(32):
    noisy = (representative_x + 0.08 * torch.randn(representative_x.shape, generator=noise_generator)).clamp(0, 1)
    noisy.requires_grad_(True)
    smooth_gradients.append(torch.autograd.grad(model(noisy)[0, representative_target], noisy)[0].abs())
smoothgrad = torch.stack(smooth_gradients).mean(0)

# A deletion test checks whether top-attributed pixels affect the chosen score.
top_pixels = smoothgrad.flatten().topk(10).indices
deleted = representative_x.clone().flatten()
deleted[top_pixels] = 0.0
deleted = deleted.view_as(representative_x)
with torch.inference_mode():
    original_score = float(model(representative_x)[0, representative_target])
    deleted_score = float(model(deleted)[0, representative_target])
assert saliency.shape == smoothgrad.shape == representative_x.shape
print({"target": representative_target, "original_logit": round(original_score, 3), "after_top10_deletion": round(deleted_score, 3)})
```

</details>

Attribution map 应接受 sanity check：随机化模型 weight 或 label，比较多种 baseline 和 seed，并把 deletion/insertion 效果与随机或 edge-based control 对照。对人类而言合理的图像不一定忠实于模型计算。


### **Grad-CAM 与 Integrated Gradients** {#grad-cam-integrated-gradients}

Grad-CAM 在卷积 feature map 中定位 target。对 channel (k) 的 activation (A^k)，先对 target gradient 做空间平均：

$$\alpha_k^c=\frac{1}{Z}\sum_{i,j}\frac{\partial y^c}{\partial A_{ij}^k}, \qquad L^c=\operatorname{ReLU}\left(\sum_k\alpha_k^cA^k\right).$$

Map 的分辨率继承自 feature map，因此比较粗糙；ReLU 只保留提高 target score 的证据。[Grad-CAM 论文](https://openaccess.thecvf.com/content_iccv_2017/html/Selvaraju_Grad-CAM_Visual_Explanations_ICCV_2017_paper.html)强调的是 class-discriminative localization，而不是 pixel-level causal segmentation。

Integrated Gradients（IG）沿 baseline (x') 到输入 (x) 的路径累积 gradient：

$$\operatorname{IG}_i(x)=(x_i-x_i')\int_0^1\frac{\partial f(x'+\alpha(x-x'))}{\partial x_i}\,d\alpha.$$

在合适条件下它满足 completeness：attribution 的总和等于 (f(x)-f(x'))。[IG 原始论文](https://proceedings.mlr.press/v70/sundararajan17a.html)说明 baseline 选择本身就是 explanation 定义的一部分。

![同一输入上的 Grad-CAM 与 Integrated Gradients 解释。](assets/dl21-gradcam-ig.png){fig-align="center" width="68%" fig-alt="同一数字旁边展示粗粒度 Grad-CAM overlay 和带符号的 Integrated Gradients pixel map。"}

<details>
<summary><strong>PyTorch：实现 Grad-CAM 并验证 Integrated Gradients completeness</strong></summary>

```python
def grad_cam(model, x, target):
    model.eval()
    input_tensor = x.detach().clone().requires_grad_(True)
    logits, parts = model.forward_intermediates(input_tensor)
    feature_map = parts["conv2"]
    feature_map.retain_grad()
    model.zero_grad()
    logits[0, target].backward()
    channel_weights = feature_map.grad.mean(dim=(2, 3), keepdim=True)
    heatmap = F.relu((channel_weights * feature_map).sum(dim=1, keepdim=True))
    heatmap = F.interpolate(heatmap, size=(8, 8), mode="bilinear", align_corners=False)
    return (heatmap / heatmap.max().clamp_min(1e-8)).detach()


def integrated_gradients(model, x, target, steps=96):
    model.eval()
    baseline = torch.zeros_like(x)
    gradients = []
    for alpha in torch.linspace(0.0, 1.0, steps + 1)[1:]:
        interpolated = (baseline + alpha * (x - baseline)).detach().requires_grad_(True)
        gradients.append(torch.autograd.grad(model(interpolated)[0, target], interpolated)[0])
    attribution = (x - baseline) * torch.stack(gradients).mean(0)
    with torch.inference_mode():
        output_difference = model(x)[0, target] - model(baseline)[0, target]
    completeness_error = float(attribution.sum() - output_difference)
    return attribution.detach(), completeness_error


cam = grad_cam(model, representative_x, representative_target)
integrated, completeness_error = integrated_gradients(model, representative_x, representative_target)
assert cam.shape == integrated.shape == representative_x.shape
assert abs(completeness_error) < 0.20
print({"target": representative_target, "IG_completeness_error": round(completeness_error, 4)})
```

</details>

Grad-CAM 询问卷积 representation 在哪里支持 target；IG 询问 target 沿选定 baseline path 如何变化，两张图不必一致。需要比较 baseline、layer、数值积分步数与 perturbation test，并把 explanation stability 当作需要评估的属性，而不是视觉假设。


### **机制与概念级可解释性** {#mechanistic-concept-interpretability}

Feature attribution 仍然绑定 pixel 或 token。Concept 方法通过正负样本定义人类层面的属性，在 activation space 中拟合方向，再询问 target output 是否沿该方向敏感。Concept activation vector（CAV）(v_C) 对应 directional derivative (S_{C,c}(x)=\nabla_h f_c(h(x))\cdot v_C)。TCAV 汇总正 sensitivity 的样本比例，并相对随机 concept 重复检验；参见 [TCAV 论文](https://proceedings.mlr.press/v80/kim18d.html)。

![从 concept example 到 activation direction 和 causal intervention。](assets/dl21-concepts.svg){fig-align="center" width="74%" fig-alt="Concept example 在 activation space 中定义方向，使用 directional derivative 测量后再进行 causal activation intervention。"}

Mechanistic interpretability 对计算提出更强主张：ablation 移除 component；activation patching 用 source example 的内部状态替换当前状态；causal tracing 定位信息在何处改变输出；circuit analysis 提出稀疏 component 与 interaction 集合。这些 intervention 仍可能离开 data manifold，且组件可能冗余，因此必须设置 control 并重复实验。

<details>
<summary><strong>PyTorch：构造 stroke-density CAV 并测量 directional sensitivity</strong></summary>

```python
train_embeddings = torch.tensor(train_representations["embedding"], dtype=torch.float32)
train_ink = images[train_ids].flatten(1).sum(1)
low_threshold, high_threshold = torch.quantile(train_ink, torch.tensor([0.25, 0.75]))
concept_mask = (train_ink <= low_threshold) | (train_ink >= high_threshold)
concept_labels = (train_ink[concept_mask] >= high_threshold).long().numpy()
concept_probe = LogisticRegression(max_iter=1000, random_state=2121)
concept_probe.fit(train_embeddings[concept_mask].numpy(), concept_labels)
cav = torch.tensor(concept_probe.coef_[0], dtype=torch.float32)
cav = cav / cav.norm().clamp_min(1e-8)

model.eval()
_, test_parts = model.forward_intermediates(id_test_x)
test_embedding = test_parts["embedding"].detach().requires_grad_(True)
logits_from_embedding = model.logits_from_embedding(test_embedding)
selected_logits = logits_from_embedding.gather(1, id_predictions[:, None]).sum()
embedding_gradient = torch.autograd.grad(selected_logits, test_embedding)[0]
concept_sensitivity = embedding_gradient @ cav
tcav_like_fraction = float((concept_sensitivity > 0).float().mean())
concept_train_accuracy = concept_probe.score(train_embeddings[concept_mask].numpy(), concept_labels)
assert cav.shape == (64,) and 0 <= tcav_like_fraction <= 1
print({"concept_probe_training_accuracy": round(concept_train_accuracy, 3), "positive_sensitivity_fraction": round(tcav_like_fraction, 3)})
```

</details>

这是教学规模的 TCAV-like 计算，不能证明网络包含稳定的“ink” concept。Stroke density 与 digit class 相关，concept example 是通过算法定义的，并且只拟合了一个方向。严谨研究需要重复 concept construction，使用匹配随机 control 和 held-out concept data，检验统计稳定性，并用 intervention 继续验证相关关系。


### **对抗样本与鲁棒性** {#adversarial-examples-robustness}

鲁棒性询问模型在指定 perturbation 下能否维持可接受行为。Natural corruption 模拟 blur、noise、compression 或 sensor missing 等采集变化；adversarial example 则在 threat-model constraint 内优化输入，以增大 loss 或迫使目标输出。在 (L_\infty) budget (\epsilon) 下，FGSM 使用 (x'=\operatorname{clip}(x+\epsilon\operatorname{sign}(\nabla_x\mathcal{L}),0,1))；projected gradient descent（PGD）重复更小的更新，并投影回允许集合。

![干净、随机噪声与 PGD 输入及其准确率。](assets/dl21-adversarial.png){fig-align="center" width="76%" fig-alt="同一数字分别以干净、随机噪声和 PGD 形式展示，柱状图比较同一 infinity-norm budget 下的 accuracy。"}

(L_p) ball 在数学上方便，却不是完整的感知或应用 threat model。Spatial transform、patch、prompt injection、poisoning、model extraction 和 physical attack 需要不同能力与 oracle。[对抗样本奠基工作](https://arxiv.org/abs/1412.6572)说明 worst-case direction 与普通随机噪声有本质差异。

<details>
<summary><strong>PyTorch：比较随机扰动与迭代 worst-case perturbation</strong></summary>

```python
def pgd_attack(model, x, y, epsilon=0.16, step_size=0.04, steps=6):
    model.eval()
    adversarial = x.detach().clone()
    for _ in range(steps):
        adversarial.requires_grad_(True)
        loss = F.cross_entropy(model(adversarial), y)
        gradient = torch.autograd.grad(loss, adversarial)[0]
        adversarial = adversarial.detach() + step_size * gradient.sign()
        adversarial = torch.max(torch.min(adversarial, x + epsilon), x - epsilon).clamp(0, 1)
    return adversarial.detach()


adversarial_x = pgd_attack(model, id_test_x, id_test_y)
random_generator = torch.Generator().manual_seed(2121)
random_noise = torch.empty(id_test_x.shape).uniform_(-0.16, 0.16, generator=random_generator)
random_x = (id_test_x + random_noise).clamp(0, 1)
with torch.inference_mode():
    random_accuracy = float(model(random_x).argmax(1).eq(id_test_y).float().mean())
    adversarial_accuracy = float(model(adversarial_x).argmax(1).eq(id_test_y).float().mean())
maximum_change = float((adversarial_x - id_test_x).abs().max())
assert maximum_change <= 0.16001 and adversarial_accuracy <= baseline_accuracy
print({"clean": round(baseline_accuracy, 3), "random": round(random_accuracy, 3), "PGD": round(adversarial_accuracy, 3), "L_inf": round(maximum_change, 3)})
```

</details>

Attack strength 是结果的一部分：必须记录 norm、budget、step、restart、target、white/black-box access 与 gradient handling。Gradient masking 会让弱 attack 显得无效。鲁棒性结论需要 adaptive attack 和独立评估；adversarial training 通常用额外计算以及有时降低 clean accuracy 的代价，换取训练 threat model 内的鲁棒性。


### **隐私与记忆** {#privacy-memorization}

Memorization 不等于 privacy harm，但可能让训练记录变得可区分。Membership inference 根据明确的 access model，使用 loss、confidence、gradient、embedding 或 generated content 判断候选记录是否参与训练。[Shokri 等人的 membership-inference 研究](https://arxiv.org/abs/1610.05820)把隐私泄漏定义为在 member 与 non-member 上评估的攻击，而不只是 train-test gap。

![从候选记录到 attack score 的 membership inference threat model。](assets/dl21-privacy.svg){fig-align="center" width="72%" fig-alt="候选 member 或 non-member 通过已发布模型接口，攻击者把输出转换为 membership guess，并使用 AUROC 评估。"}

Differentially private SGD 限制单条记录变化时训练算法输出分布的变化。逐样本 gradient 被裁剪到 norm (C)，求和后加入噪声：

$$\tilde g=\frac{1}{B}\left(\sum_{i=1}^{B}g_i\min\left(1,\frac{C}{\|g_i\|_2}\right)+\mathcal{N}(0,\sigma^2C^2I)\right).$$

隐私结论需要给定 sampling scheme 与 accountant，并报告 ((\epsilon,\delta))；仅有 clipping 和 noise 还没有说明 guarantee。隐私还包括 data minimization、access control、retention、deletion、secure log 与 output restriction。

<details>
<summary><strong>PyTorch：审计 loss-threshold membership attack</strong></summary>

```python
audit_size = min(len(test_ids), len(train_ids))
member_x = images[train_ids[:audit_size]]
member_y = targets[train_ids[:audit_size]]
nonmember_x = id_test_x[:audit_size]
nonmember_y = id_test_y[:audit_size]
with torch.inference_mode():
    member_loss = F.cross_entropy(model(member_x), member_y, reduction="none")
    nonmember_loss = F.cross_entropy(model(nonmember_x), nonmember_y, reduction="none")

membership_labels = np.concatenate([np.ones(audit_size), np.zeros(audit_size)])
membership_scores = torch.cat([-member_loss, -nonmember_loss]).numpy()
membership_auc = roc_auc_score(membership_labels, membership_scores)
train_loss_mean = float(member_loss.mean())
test_loss_mean = float(nonmember_loss.mean())
assert 0 <= membership_auc <= 1
print({"attack_AUROC": round(membership_auc, 3), "member_loss": round(train_loss_mean, 3), "nonmember_loss": round(test_loss_mean, 3)})
```

</details>

AUROC 接近 0.5 只说明在当前 sampling 和 interface 下该 attack 失败，不是 privacy proof。更强的 audit 会按 class 与 difficulty 匹配记录，考虑多种 attack，并说明 auxiliary knowledge。生成系统还需要测试 canary exposure、verbatim regurgitation、nearest-neighbor overlap 与重复查询下的 extraction。


### **公平性与 Subgroup 评估** {#fairness-subgroup-evaluation}

公平性是 system、population、decision 与 harm 共同构成的 socio-technical property。Demographic parity、equalized odds、equal opportunity、predictive parity 与 group calibration 编码了不同规范目标，在 base rate 不同时还可能冲突。正确问题不是“模型是否公平”，而是谁受到影响、分配了什么 outcome、哪类 error 造成 harm，以及存在何种 intervention。

![以 context、group、metric 和 action 为基础的公平性评估流程。](assets/dl21-fairness.svg){fig-align="center" width="74%" fig-alt="流程从受影响背景经过有意义 group 和 metric 进入 mitigation，同时警告 operational slice 不能证明 demographic fairness。"}

Subgroup evaluation 为预先指定和 intersectional slice 计算 metric，给出样本量与不确定性，并报告具有足够样本支持的 worst group，而不只是平均值。小 group 的估计不稳定，大量探索 slice 会产生 multiple-comparison risk。Label 与 protected attribute 本身也可能缺失、带噪、由社会构造，甚至不适合收集。

<details>
<summary><strong>Python：审计 class 与 stroke-density slice，但不做人口公平性主张</strong></summary>

```python
# Thresholds are estimated from training images only.
training_density = images[train_ids].flatten(1).sum(1)
density_thresholds = torch.quantile(training_density, torch.tensor([0.25, 0.75]))
test_density = id_test_x.flatten(1).sum(1)
density_group = torch.bucketize(test_density, density_thresholds)


def grouped_accuracy(prediction, label, group):
    result = {}
    for value in torch.unique(group):
        mask = group == value
        result[int(value)] = {"count": int(mask.sum()), "accuracy": float(prediction[mask].eq(label[mask]).float().mean())}
    return result


density_metrics = grouped_accuracy(id_predictions, id_test_y, density_group)
class_metrics = grouped_accuracy(id_predictions, id_test_y, id_test_y)
worst_density_accuracy = min(item["accuracy"] for item in density_metrics.values())
worst_class_accuracy = min(item["accuracy"] for item in class_metrics.values())
assert sum(item["count"] for item in density_metrics.values()) == len(id_test_y)
print({
    "density_groups": {key: {"n": value["count"], "acc": round(value["accuracy"], 3)} for key, value in density_metrics.items()},
    "worst_density_accuracy": round(worst_density_accuracy, 3),
    "worst_class_accuracy": round(worst_class_accuracy, 3),
})
```

</details>

Stroke density 是可观察的 operational slice，不是 protected attribute，也不是 demographic fairness 证据。UCI Digits 缺少得出该结论所需的社会背景。这项限制必须写入 model card，并在任何影响人的应用之前触发收集或选择 fit-for-purpose evaluation data。


### **数据来源与文档记录** {#data-provenance-documentation}

Provenance 让结果可以追溯到 source data、license、version、transformation、split membership、code、configuration、environment 与 model artifact。可变 URL 或 dataset name 还不够：上游内容可能变化，而实验标签保持不变。Hash 可以识别准确 bytes，却无法解释 collection consent、representativeness、annotation decision 或 known harm。

![从数据来源和快照到 transformation 与 model artifact 的 lineage graph。](assets/dl21-provenance.svg){fig-align="center" width="75%" fig-alt="流程把 source DOI 和 license 连接到 snapshot checksum、split transformation、code run 以及带指标和审批的最终 model artifact。"}

Datasheet 与 Data Card 记录 dataset motivation、composition、collection、preprocessing、use、distribution、maintenance 和 ethical consideration。[Data Cards 工作](https://research.google/pubs/data-cards-purposeful-and-transparent-dataset-documentation-for-responsible-ai/)把文档视为服务下游读者的产品，而不是建模结束后填写的表格。Lineage system 既要足够 machine-readable 以复现 split，也要足够 human-readable 以暴露 limitation。

<details>
<summary><strong>Python：为本章实验构建带 checksum 的 lineage manifest</strong></summary>

```python
def sha256_array(array):
    contiguous = np.ascontiguousarray(array)
    return hashlib.sha256(contiguous.tobytes()).hexdigest()


model_buffer = io.BytesIO()
torch.save(model.state_dict(), model_buffer)
lineage_manifest = {
    "dataset": {
        "name": "UCI Optical Recognition of Handwritten Digits",
        "doi": "10.24432/C50P49",
        "license": "CC BY 4.0",
        "feature_hash": sha256_array(digits.data),
        "target_hash": sha256_array(digits.target),
    },
    "split": {
        "unit": "image row",
        "seed": 2121,
        "train_ids_hash": sha256_array(train_ids),
        "validation_ids_hash": sha256_array(val_ids),
        "test_ids_hash": sha256_array(test_ids),
        "ood_definition": "all rows with label 9; excluded from fitting",
    },
    "environment": {"torch": torch.__version__, "sklearn": sklearn.__version__},
    "artifact_sha256": hashlib.sha256(model_buffer.getvalue()).hexdigest(),
}
assert len({lineage_manifest["split"][key] for key in ("train_ids_hash", "validation_ids_hash", "test_ids_hash")}) == 3
assert lineage_manifest["dataset"]["license"] == "CC BY 4.0"
print(json.dumps(lineage_manifest, indent=2)[:900])
```

</details>

Manifest 是必要但不充分的记录。完整资料还要保留 acquisition/annotation context、data-removal obligation、preprocessing code、evaluator identity、approval 与 incident history。Provenance 应在 artifact 从 notebook 晋升到 registry 再进入 serving environment 的过程中保持完整。


### **Red Teaming 与安全评估** {#red-teaming-safety-evaluation}

Red teaming 从有动机 actor 或有害运行条件的角度搜索失败。首先定义 threat model：actor、access、knowledge、objective、protected asset 与可容忍 impact；然后根据系统覆盖 malformed input、boundary value、distribution shift、evasion、poisoning、privacy extraction、unsafe capability、prompt/tool abuse 与 resource exhaustion 等测试。

![由 threat model 驱动的 red teaming，以及把失败转化为 regression test。](assets/dl21-red-team.svg){fig-align="center" width="75%" fig-alt="Threat model 选择 attack，behavioral oracle 判断结果，修复后重新测试，每个失败都进入 regression suite。"}

安全评估需要比“模型应表现良好”更具体的 oracle。预期行为可以是 reject、abstain、限制资源、保持 policy、请求人工复核或记录 incident。[NIST AI Risk Management Framework](https://www.nist.gov/itl/ai-risk-management-framework)把持续工作组织为 govern、map、measure 与 manage；它是一套风险管理过程，而不是“单个 benchmark 可以认证安全”的主张。

<details>
<summary><strong>Python：把 malformed 与 shifted request 转换为 regression harness</strong></summary>

```python
def guarded_predict(raw_pixels):
    array = np.asarray(raw_pixels, dtype=np.float32)
    if array.shape not in {(8, 8), (1, 8, 8)}:
        raise ValueError("expected one 8 by 8 image")
    if not np.isfinite(array).all() or array.min() < 0 or array.max() > 16:
        raise ValueError("pixels must be finite values in 0..16")
    tensor = torch.from_numpy(array.reshape(1, 1, 8, 8) / 16.0)
    with torch.inference_mode():
        logits = model(tensor)
        energy = float(-torch.logsumexp(logits, dim=1))
        confidence, prediction = logits.softmax(1).max(1)
    return {"prediction": int(prediction), "confidence": float(confidence), "abstain": energy > ood_energy_threshold}


clean_request = digits.images[int(test_ids[0])]
test_cases = {
    "clean": clean_request,
    "blank": np.zeros((8, 8), dtype=np.float32),
    "inverted": 16.0 - clean_request,
    "malformed": np.zeros((16, 16), dtype=np.float32),
    "out_of_range": np.full((8, 8), 17.0, dtype=np.float32),
}
red_team_results = {}
for name, request in test_cases.items():
    try:
        red_team_results[name] = {"accepted": True, **guarded_predict(request)}
    except ValueError as error:
        red_team_results[name] = {"accepted": False, "reason": str(error)}

assert red_team_results["clean"]["accepted"]
assert not red_team_results["malformed"]["accepted"] and not red_team_results["out_of_range"]["accepted"]
print(red_team_results)
```

</details>

Red-team campaign 记录 coverage 和 unresolved risk，而不只是成功 attack。需要把 failure discoverer 与 fix owner 分开，保护敏感 exploit detail，在 model/runtime 变化后重新运行测试，并把每个确认失败转换为有版本的 regression case。Generative 与 agentic system 还需要针对 content、instruction hierarchy、tool authorization、long-horizon behavior 与 human overreliance 的领域测试。


### **Model Card 与负责任发布** {#model-cards-responsible-release}

Model card 与某个确定 artifact 同时发布，说明模型是什么、由谁负责、intended/out-of-scope use、training/evaluation data、metric 与 slice、limitation、ethical consideration、runtime requirement 以及 monitoring/rollback expectation。[Model Cards 提案](https://research.google/pubs/model-cards-for-model-reporting/)强调 context 与 disaggregated performance，而不是一段 leaderboard 描述。

![Model card 中的 scope、evidence、control 与 release decision。](assets/dl21-model-card.svg){fig-align="center" width="74%" fig-alt="三列分别记录 model scope、evaluation evidence 和 operational control，最终形成 approved、restricted 或 blocked release decision。"}

负责任发布是一项决策过程。Evidence owner 在最终测试前定义 gate；risk owner 对 unresolved risk 进行接受、缓解、转移或避免；change management 规定何时必须重新评估。Open release、gated access、API-only access、staged deployment 与 non-release 是不同选项。文档不能把不适合的模型变成可接受模型。

<details>
<summary><strong>Python：把证据汇总为包含明确 blocker 的 release card</strong></summary>

```python
release_metrics = {
    "ID_accuracy": baseline_accuracy,
    "calibrated_ECE": calibrated_ece,
    "energy_OOD_AUROC": energy_ood_auc,
    "PGD_accuracy_epsilon_0.16": adversarial_accuracy,
    "membership_attack_AUROC": membership_auc,
    "worst_density_slice_accuracy": worst_density_accuracy,
}
release_gates = {
    "ID_accuracy": release_metrics["ID_accuracy"] >= 0.93,
    "calibration": release_metrics["calibrated_ECE"] <= 0.08,
    "OOD_detection": release_metrics["energy_OOD_AUROC"] >= 0.70,
    "adversarial_robustness": release_metrics["PGD_accuracy_epsilon_0.16"] >= 0.50,
    "membership_audit": release_metrics["membership_attack_AUROC"] <= 0.60,
    "operational_slice": release_metrics["worst_density_slice_accuracy"] >= 0.85,
}
blockers = [name for name, passed in release_gates.items() if not passed]
model_card = {
    "model": "TinyDigitCNN-v1",
    "intended_use": "teaching-scale evaluation of digits 0 through 8",
    "out_of_scope": ["human-impacting decisions", "wild handwriting deployment", "demographic fairness claims"],
    "data": lineage_manifest["dataset"],
    "metrics": {name: round(value, 4) for name, value in release_metrics.items()},
    "limitations": ["digit 9 is the only semantic OOD family", "no demographic attributes", "single seed and small dataset"],
    "release_status": "restricted teaching artifact" if blockers else "approved for intended use",
    "blockers_for_broader_use": blockers,
}
assert model_card["out_of_scope"] and model_card["release_status"]
print(json.dumps(model_card, indent=2))
```

</details>

即使所有数值 gate 都通过，release status 仍应保持狭窄，因为这些 gate 只覆盖当前 dataset 与 threat model。生产 model card 还要加入 accountable owner、uncertainty interval、subgroup definition、runtime profile、human-factors evaluation、security review、法律/组织要求，以及 incident 与 rollback procedure 链接。


### **章节对比与总结** {#chapter-comparison-summary}

Trustworthiness 不是一个标量，可解释性也不是一张图片。每种方法都只回答有边界的问题，并具有典型失败模式。

| 方法 | 回答的问题 | 产生的证据 | 常见过度主张 |
|---|---|---|---|
| Task 与 slice metric | 模型多久成功、在哪里失败？ | aggregate、class、subgroup 与 selective risk | 一个平均值代表所有用户 |
| Calibration / uncertainty | confidence 能否支持决策策略？ | reliability、proper score、disagreement | softmax entropy 等于 epistemic uncertainty |
| OOD detection | 选定 shift 能否与 ID 输入分离？ | AUROC、FPR@TPR、threshold behavior | 一个 OOD 数据集证明 open-world detection |
| Probe | 属性能否从 representation 中解码？ | held-out controlled readout | decodability 证明 causal use |
| Saliency / Grad-CAM / IG | 在既定定义下，哪些局部 feature 影响 target？ | attribution map 与 perturbation test | 合理图像解释整个模型 |
| TCAV / intervention | output 是否对 concept 或 internal state 敏感？ | directional test、ablation、patching | 一个 concept direction 就识别了 circuit |
| Robustness evaluation | 指定 perturbation/threat model 下会发生什么？ | corruption curve 与 adaptive attack | 一个 norm 或 attack 证明 security |
| Privacy audit | 明确 attacker 能否推断敏感训练影响？ | 定义 access 下的 attack advantage | 一次失败 attack 证明 privacy |
| Fairness evaluation | use context 中 error/outcome 如何分布？ | 有样本支持的 subgroup metric 与 harm | 任意方便 slice 都能证明 fairness |
| Provenance / card / red team | 证据、限制、责任与响应能否追溯？ | lineage、regression test、release decision | documentation 可以代替 mitigation |

<details>
<summary><strong>Python：验证每项 release claim 都有证据、边界与 owner</strong></summary>

```python
evidence_registry = [
    {"claim": "ID task quality", "evidence": "test metrics and slices", "boundary": "digits 0-8 fixed split", "owner": "model evaluation"},
    {"claim": "confidence policy", "evidence": "validation temperature and test reliability", "boundary": "current class prior", "owner": "risk evaluation"},
    {"claim": "OOD routing", "evidence": "digit-9 energy AUROC and threshold", "boundary": "one semantic OOD family", "owner": "serving"},
    {"claim": "local explanation", "evidence": "saliency, Grad-CAM, IG, deletion and completeness", "boundary": "selected targets/baselines", "owner": "interpretability"},
    {"claim": "adversarial robustness", "evidence": "PGD under recorded budget", "boundary": "white-box L-inf threat", "owner": "security"},
    {"claim": "privacy risk", "evidence": "loss-threshold membership audit", "boundary": "black-box loss score", "owner": "privacy"},
    {"claim": "responsible release", "evidence": "lineage, model card, red-team regression", "boundary": "teaching use only", "owner": "release authority"},
]
required_fields = {"claim", "evidence", "boundary", "owner"}
assert all(set(record) == required_fields and all(record.values()) for record in evidence_registry)
print({"registered_claims": len(evidence_registry), "all_bounded_and_owned": True})
```

</details>

一条可辩护的工作流会先定义决策与 harm，冻结数据边界，评估总体和 slice，校准 uncertainty，测试 shift 与 attack，使用受控方法研究机制，在明确限制下审计 privacy 与 fairness，记录 provenance，并把失败连接到 owner 与 rollback。数据、模型、运行时、policy 或 use context 变化后，都必须更新证据。
